# เบื้องหลังหน้าเว็บ: เรียกตลาดด้วย Python

ทุกปุ่มในหน้าเว็บคือ **การเรียก API** หนึ่งครั้ง แบบฝึกนี้ให้ทีม **เรียก API เอง** อ่านคำตอบที่ตลาดส่งกลับมา แล้วเล่นตลาดฝึกหนึ่งเกมโดยไม่กดปุ่มในหน้าเว็บ

**วิธีทำ:** แทน `___` ด้วยโค้ดที่ถูก แล้วรันช่องนั้น (Shift+Enter) ทำจากบนลงล่าง · ส่วนที่ไม่มี `___` เขียนไว้ให้แล้ว อ่านคอมเมนต์ (`#`) แล้วรันได้เลย

| ทำช่วงไหน | ข้อ |
|---|---|
| 14:45–15:15 ฟังอธิบาย API แล้วตั้งค่า ดูตลาดครั้งแรก | 1–2 |
| 15:15–17:00 เล่นตลาดฝึกด้วย Python หนึ่งเกม (ไม่นับคะแนน) | 3–11 |

**รอบเก็บคะแนนช่วง 1 (13:15–14:45) ขายผ่านหน้าเว็บ `/trade` ด้วยมือเท่านั้น** ห้ามใช้ notebook นี้หรือสคริปต์ใดๆ ส่งคำขอ — ถ้าใช้ คะแนนช่วง 1 เป็น 0 (เซิร์ฟเวอร์บันทึกว่าคำขอมาจากโปรแกรมอะไร) · notebook นี้ใช้กับ **ตลาดฝึก** เท่านั้น

**ปุ่มไหน เรียก API อะไร** (ทุกช่องเขียน path ไว้ให้แล้ว)

| ในหน้าเว็บ | API | ข้อ |
|---|---|---|
| เชื่อมต่อได้ไหม | `GET /v1/version` | 1 |
| สถานะ ล็อต คำขอซื้อ | `GET /v1/state` | 2, 7 |
| เริ่มใหม่ (ตลาดฝึก) | `POST /v1/sandbox/reset` | 3 |
| กล่องสูตรมูลค่า | `GET /v1/calendar` | 4 |
| ส่งผลประเมิน | `POST /v1/appraisals` | 5 |
| ยื่นซอง | `POST /v1/offers` | 6, 8, 10 |
| ข้อเสนอที่เปิดอยู่ | `GET /v1/offers` | 8 |
| ผลรอบ | `GET /v1/results` | 9 |
| ดูมูลค่าจริง (ตลาดฝึก) | `GET /v1/sandbox/reveal` | 11 |

**เขียนอย่างไร:** `GET` = `api.get("/path")` · `POST` = `api.post("/path", json=ข้อมูล)` · ต่อท้าย `.json()` เพื่อได้ dict ของ Python

## 1. ตั้งค่า แล้วเรียก API ครั้งแรก

แก้สองบรรทัดแรกของช่องข้างล่าง: ที่อยู่ตลาดฝึก (`MARKET`) และ key ฝึกของทีม (`KEY` ตัวเดียวกับที่ใส่ในหน้าเว็บ) **อย่าแชร์ key ให้ทีมอื่น** ถ้ายังไม่ได้แก้ ช่องนี้จะบอกเป็นภาษาไทยว่าต้องแก้อะไร

แล้วรันช่องนี้ ช่องนี้เรียก **เชื่อมต่อได้ไหม** · ถ้าขึ้น `No module named 'httpx'` ให้รัน `pip install httpx` ใน Terminal ก่อน

**คำตอบของ API มีสองส่วน**

- `r.status_code` เลขบอกผล: `200` สำเร็จ · `201` สร้างสำเร็จ (เช่น ยื่นซองแล้ว) · `4xx` คำขอผิด (ข้อ 6) · `5xx` เซิร์ฟเวอร์มีปัญหา
- `r.json()` ข้อมูลที่ตอบกลับ แปลงเป็น dict / list ของ Python แล้ว

ทุกข้อใน notebook นี้จะ **ดูคำตอบเต็มก่อน** ด้วยฟังก์ชัน `show()` แล้วค่อยหยิบช่องที่ใช้ จะได้รู้ว่าแต่ละ API ส่งอะไรกลับมาบ้าง

**notebook ไม่มีจอบอกช่วงของตลาด** ช่องนี้จึงสร้าง `where()` ไว้ด้วย ช่องต่อจากนี้รัน `where()` เป็นบรรทัดแรก พิมพ์บรรทัดเดียว เช่น `[T05 · ส่งผลประเมินได้ (APPRAISAL) · เหลือ 412 วินาที]` · ไม่แน่ใจเมื่อไรรัน `where()` ในช่องใหม่ได้เลย

เปิดหน้าเว็บ `/trade` ของ **ตลาดฝึก (ที่อยู่เดียวกับ MARKET)** ด้วย key เดียวกันไว้ข้างๆ ได้ จะเห็นตลาดเดียวกัน ถ้าล็อตไม่ตรงกัน แปลว่าหน้าเว็บเปิดคนละที่อยู่ (เช่น พอร์ต 8001 ของรอบเก็บคะแนน) หรือใส่คนละ key

In [49]:
import json
import httpx

MARKET = "http://202.29.227.22:8000"   # ที่อยู่ตลาดฝึก ทีมงานแจกในห้อง (ตลาดฝึกใช้พอร์ต 8000)
KEY = "sk-T01-3LMFDCSPM36y1gs53bFrZCL7"           # key ฝึกของทีม ขึ้นต้นด้วย sk-T

# ตรวจก่อนว่าแก้สองบรรทัดบนแล้ว (ถ้าไม่ตรวจ จะได้ error ภาษาอังกฤษที่อ่านยาก)
if "<" in MARKET:
    raise ValueError("ยังไม่ได้ใส่ที่อยู่ตลาด: แก้ MARKET ในช่องนี้ เช่น http://192.168.1.10:8000")
if not KEY.isascii() or not (KEY.startswith("sk-T") or KEY == "demo-team-key") or KEY.startswith("sk-T00-"):
    raise ValueError("ยังไม่ได้ใส่ key ของทีม: แก้ KEY ในช่องนี้เป็น key ที่ทีมงานแจก (ขึ้นต้นด้วย sk-T ไม่มีภาษาไทย)")

class Blank:
    """ช่อง ___ ที่ยังไม่ได้เติม: ถ้าลืมเติม จะได้ error ภาษาไทยแทนการส่งค่าแปลกๆ ไปตลาด"""
    def __repr__(self):
        return "<ช่อง ___ ยังไม่ได้เติม>"
    def _stop(self, *args):
        raise ValueError("ยังไม่ได้เติมช่อง ___ : แทน ___ ด้วยโค้ด แล้วรันช่องนั้นใหม่")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = __truediv__ = __rtruediv__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __round__ = __float__ = __int__ = __index__ = _stop

___ = Blank()   # Jupyter ใช้ชื่อ ___ เก็บผลลัพธ์เก่า ตั้งเองไว้ก่อน ช่องที่ลืมเติมจะได้ไม่ส่งผลลัพธ์เก่าไป

class Market(httpx.Client):
    """httpx.Client ที่ตรวจก่อนส่งว่า body ไม่มีช่อง ___ ค้างอยู่"""
    def request(self, method, url, **kwargs):
        if "<ช่อง ___ ยังไม่ได้เติม>" in repr(kwargs.get("json")):
            Blank()._stop()
        return super().request(method, url, **kwargs)

# api คือตัวเรียกตลาด สร้างครั้งเดียว ใช้ทุกข้อต่อจากนี้
#   base_url  ที่อยู่ตลาด ต่อไปเขียนแค่ path เช่น "/v1/state"
#   headers   แนบ key ไปกับทุกคำขอ ตลาดใช้ key รู้ว่าเราเป็นทีมไหน
#   timeout   รอคำตอบไม่เกิน 10 วินาที
api = Market(base_url=MARKET, headers={"Authorization": f"Bearer {KEY}"}, timeout=10)

PHASE_TH = {"WAITING": "รอเริ่ม", "SETUP": "เตรียมตัว", "APPRAISAL": "ส่งผลประเมินได้",
            "ROUND_OPEN": "รอบเปิดรับข้อเสนอ", "ROUND_CLEARING": "กำลังจับคู่", "CLOSED": "ปิดตลาด"}

def where():
    """บอกว่าตลาดอยู่ช่วงไหน แทนจอ: ทีม · ช่วง · รอบ · เหลือกี่วินาที (เรียก GET /v1/state)"""
    s = api.get("/v1/state").json()
    if "round" not in s:
        raise RuntimeError(f"GET /v1/state returned an error: {json.dumps(s, ensure_ascii=False)[:500]}")
    rnd = f" · รอบ {s['round']['number']}/{s['round']['of']}" if s["round"] else ""
    left = f" · เหลือ {s['seconds_left']:.0f} วินาที" if s["seconds_left"] is not None else ""
    print(f"[{s['team_id']} · {PHASE_TH.get(s['phase'], s['phase'])} ({s['phase']}){rnd}{left}]")
    return s

def show(data):
    """พิมพ์ JSON ให้อ่านง่าย: ย่อหน้าตามชั้น และแสดงภาษาไทยเป็นตัวอักษร (ensure_ascii=False)"""
    print(json.dumps(data, ensure_ascii=False, indent=2))

r = api.get("/v1/version")
print(r.status_code)   # ต้องได้ 200
show(r.json())
# api          รุ่นของ API (v1)
# server       รุ่นของโปรแกรมตลาด
# mode         sandbox = ตลาดฝึก
# server_time  เวลาของเซิร์ฟเวอร์ เป็นวินาทีนับจาก 1 ม.ค. 1970
# trade_page   ตลาดนี้เปิดหน้าเว็บ /trade ไหม

where()   # ตลาดอยู่ช่วงไหน

200
{
  "api": "v1",
  "server": "0.1.0",
  "mode": "sandbox",
  "server_time": 1794975467.7139516,
  "trade_page": true
}
[T01 · ส่งผลประเมินได้ (APPRAISAL) · เหลือ 8 วินาที]


{'server_time': 1794975467.8909178,
 'market': 'sandbox',
 'team_id': 'T01',
 'phase': 'APPRAISAL',
 'paused': False,
 'phase_ends_at': 1794975475.4604688,
 'seconds_left': 7.6,
 'round': None,
 'lots': [{'lot_id': 'LOT-T01-1',
   'kg_total': 420,
   'kg_remaining': 420,
   'kg_in_open_offers': 0,
   'kg_available': 420,
   'origin': 'สุราษฎร์ธานี - ท่าชนะ',
   'process': 'Washed',
   'sample': {'defect_pct': 1.8, 'moisture_pct': 11.9, 'screen': 12}},
  {'lot_id': 'LOT-T01-2',
   'kg_total': 540,
   'kg_remaining': 540,
   'kg_in_open_offers': 0,
   'kg_available': 540,
   'origin': 'สุราษฎร์ธานี - ท่าชนะ',
   'process': 'Honey',
   'sample': {'defect_pct': 3.5, 'moisture_pct': 12.7, 'screen': 14}},
  {'lot_id': 'LOT-T01-3',
   'kg_total': 480,
   'kg_remaining': 480,
   'kg_in_open_offers': 0,
   'kg_available': 480,
   'origin': 'ระนอง - กระบุรี',
   'process': 'Natural (Dry)',
   'sample': {'defect_pct': 5.5, 'moisture_pct': 13.7, 'screen': 18}},
  {'lot_id': 'LOT-T01-4',
   'kg_tot

## 2. ตลาดตอนนี้เป็นอย่างไร

คำขอ **สถานะ** ได้ข้อมูลทุกอย่างที่หน้าเว็บแสดงในครั้งเดียว คำตอบยาว จึงดูชื่อช่องระดับบนก่อน แล้วค่อยเจาะทีละส่วน

| ช่อง | ความหมาย |
|---|---|
| `phase` | ช่วงของตลาด (ตารางล่าง) |
| `seconds_left`, `phase_ends_at` | เหลือกี่วินาทีจะจบช่วงนี้ และเวลาที่จะจบ (เวลาของเซิร์ฟเวอร์) |
| `round` | รอบปัจจุบัน: `round_id`, `number`, `of` (ทั้งหมดกี่รอบ) · `null` ถ้ายังไม่ถึงช่วงขาย |
| `lots` | ล็อตของทีม (ตารางล่าง) |
| `rfqs` | คำขอซื้อของรอบนี้ ว่างถ้ารอบยังไม่เปิด (ดูข้อ 7) |
| `my_open_offers` | ซองที่ทีมยื่นและยังเปิดอยู่ |
| `round_capacity_kg`, `capacity_left_kg` | เพดานกระสอบรวมต่อรอบ วันนี้เป็น `null` = ไม่จำกัด (ตลาดอื่นอาจมีเพดาน ดูที่ช่องนี้) |
| `appraisal_submitted` | ส่งผลประเมินแล้วหรือยัง |
| `last_results` | ผลรอบล่าสุด (แบบเดียวกับข้อ 9) |
| `standings` | ยอดของทีม: รายได้ (`revenue_thb`), ขายแล้ว / เหลือ / ทั้งหมดกี่ กก., ราคาเฉลี่ยที่ขายได้ |
| `salvage_ratio` | ของที่ขายไม่ออกตอนจบ ได้ 0.8 × มูลค่าจริง |
| `server_time`, `market`, `team_id`, `paused` | เวลาเซิร์ฟเวอร์ · ชื่อตลาด · ทีมเรา · ตลาดหยุดชั่วคราวไหม (ใช้ในวันแข่ง) |

| phase | ความหมาย |
|---|---|
| `SETUP` | เตรียมตัว |
| `APPRAISAL` | ส่งผลประเมินได้ |
| `ROUND_OPEN` | รอบเปิดรับข้อเสนอ |
| `ROUND_CLEARING` | กำลังจับคู่ |
| `CLOSED` | ปิดตลาด |

In [50]:
r = api.get("/v1/state")
state = r.json()
print(r.status_code)
print(list(state))   # ชื่อช่องระดับบนทั้งหมด เทียบกับตารางด้านบน

200
['server_time', 'market', 'team_id', 'phase', 'paused', 'phase_ends_at', 'seconds_left', 'round', 'lots', 'rfqs', 'my_open_offers', 'round_capacity_kg', 'capacity_left_kg', 'appraisal_submitted', 'last_results', 'standings', 'salvage_ratio']


เจาะดู **ล็อตแรกแบบเต็ม** · ช่องของล็อต:

| ช่อง | ความหมาย |
|---|---|
| `lot_id` | รหัสล็อต ใช้ตอนส่งผลประเมินและยื่นซอง |
| `kg_total` | ล็อตนี้มีทั้งหมดกี่ กก. (1 กระสอบ = 60 กก.) |
| `kg_remaining` | ยังขายไม่ออกกี่ กก. |
| `kg_in_open_offers` | กี่ กก. ถูกจองอยู่ในซองที่ยังเปิด |
| `kg_available` | ยื่นเพิ่มได้อีกกี่ กก. (= เหลือ − จองในซอง) |
| `origin`, `process` | แหล่งปลูก และวิธีแปรรูป |
| `sample` | ผลวัดตัวอย่าง: ข้อบกพร่อง % (`defect_pct`), ความชื้น % (`moisture_pct`), ขนาดเมล็ด (`screen`) ใช้คิด TV ในข้อ 4 |

In [51]:
show(state["lots"][0])

print("ช่วง:", state["phase"], "· เหลือ", state["seconds_left"], "วินาที")
for lot in state["lots"]:
    print(lot["lot_id"], lot["sample"], "พร้อมขาย", lot["kg_available"] // 60, "กระสอบ")

{
  "lot_id": "LOT-T01-1",
  "kg_total": 420,
  "kg_remaining": 420,
  "kg_in_open_offers": 0,
  "kg_available": 420,
  "origin": "สุราษฎร์ธานี - ท่าชนะ",
  "process": "Washed",
  "sample": {
    "defect_pct": 1.8,
    "moisture_pct": 11.9,
    "screen": 12
  }
}
ช่วง: APPRAISAL · เหลือ 7.5 วินาที
LOT-T01-1 {'defect_pct': 1.8, 'moisture_pct': 11.9, 'screen': 12} พร้อมขาย 7 กระสอบ
LOT-T01-2 {'defect_pct': 3.5, 'moisture_pct': 12.7, 'screen': 14} พร้อมขาย 9 กระสอบ
LOT-T01-3 {'defect_pct': 5.5, 'moisture_pct': 13.7, 'screen': 18} พร้อมขาย 8 กระสอบ
LOT-T01-4 {'defect_pct': 0.8, 'moisture_pct': 11.9, 'screen': 16} พร้อมขาย 8 กระสอบ
LOT-T01-5 {'defect_pct': 17.6, 'moisture_pct': 10.7, 'screen': 15} พร้อมขาย 7 กระสอบ


ถ้าเห็นล็อตของทีมครบ 4 ล็อต แปลว่าพร้อมทำข้อ 3 แล้ว

## 3. เริ่มเกมใหม่ (รอสัญญาณจากผู้สอน)

เหมือนกดปุ่ม **"เริ่มใหม่"** ในหน้าเว็บ แต่เขียนเป็นคำขอเอง ได้ล็อตชุดใหม่ และเวลาประเมิน 10 นาทีเริ่มนับทันที

คำขอนี้ **ส่ง** ข้อมูล ไม่ใช่ขอดู (ไม่มี body) · คำตอบคือ **สถานะของเกมใหม่** หน้าตาเหมือนคำตอบของข้อ 2

In [52]:
r = api.post("/v1/sandbox/reset")
state = r.json()                     # คำตอบเป็นสถานะ ช่องเหมือนข้อ 2
print(r.status_code, "ช่วง:", state["phase"])
for lot in state["lots"]:            # ล็อตชุดใหม่ lot_id เดิมแต่ผลวัดเปลี่ยน
    print(lot["lot_id"], lot["sample"])

200 ช่วง: SETUP
LOT-T01-1 {'defect_pct': 1.1, 'moisture_pct': 11.3, 'screen': 14}
LOT-T01-2 {'defect_pct': 2.8, 'moisture_pct': 11.5, 'screen': 18}
LOT-T01-3 {'defect_pct': 3.6, 'moisture_pct': 12.9, 'screen': 14}
LOT-T01-4 {'defect_pct': 5.8, 'moisture_pct': 13.3, 'screen': 16}
LOT-T01-5 {'defect_pct': 14.1, 'moisture_pct': 12.4, 'screen': 12}


## 4. คำนวณมูลค่าด้วย Python (สูตรเดียวกับใน Excel)

ตัวเลขของสูตรอยู่ใน **กล่องสูตรมูลค่า** ช่องนี้ดึงมา คำตอบมีสองส่วน: `calendar` และ `rules` ดูทั้งหมดก่อน

| ส่วน | ช่อง | ความหมาย |
|---|---|---|
| `calendar` | `rounds`, `label` | ปฏิทินตลาด: รอบไหนเป็นช่วงอะไร (วันนี้ทุกรอบ "ปกติ") |
| | `demand_weight` | ผู้ซื้ออยากได้มากน้อยเท่าไรเทียบกับปกติ (1.0 = ปกติ) |
| | `specialty_share` | สัดส่วนคำขอซื้อของผู้ซื้อเกรดพิเศษในรอบนั้น `null` = ตามปกติ |
| `rules` | `p0` | ราคาฐาน บาท/กก. |
| | `defect_steps` | ขั้นเกรดตามข้อบกพร่อง: ไม่เกิน `max_defect_pct` ได้เกรด `grade` และตัวคูณ `step` |
| | `moisture` | ตัวคูณความชื้น: ไม่เกิน `ref` ได้ 1 · เกินแล้วลดตามสูตร · เกิน `hard_limit` ได้ `hard_factor` |
| | `screen_factors` | ตัวคูณตามขนาดเมล็ด |
| | `true_value_formula`, `G`, `M` | สูตรเขียนเป็นข้อความ ไว้อ่าน |
| | `bag_kg` | 1 กระสอบ = 60 กก. |
| | `salvage_ratio` | ของเหลือตอนจบ ได้ 0.8 × มูลค่าจริง |
| | `round_capacity`, `round_capacity_factor` | เพดานกระสอบต่อรอบ วันนี้ไม่มี (`null`) |
| | `scoring` | คะแนน: PR เต็ม `pr_points` · ผลประเมินเต็ม `appraisal_points` เหลือ 0 เมื่อคลาดเฉลี่ย (MAPE) ถึง `appraisal_zero_at_mape` (0.1 = 10%) · `leaderboard_freeze_after_round` อันดับหยุดอัปเดตหลังรอบนี้ (99 = ไม่หยุด) |
| | `timeline` | ความยาวแต่ละช่วงเป็นวินาที (`appraisal_s`, `open_s`, `clearing_s`) และจำนวนรอบ (`rounds`) |

In [53]:
where()
cal = api.get("/v1/calendar").json()
print(list(cal))          # ['calendar', 'rules']
show(cal)                 # ทั้งหมด เทียบกับตารางด้านบน

[T01 · เตรียมตัว (SETUP) · เหลือ 5 วินาที]
['calendar', 'rules']
{
  "calendar": [
    {
      "rounds": "1-8",
      "label": "ปกติ",
      "demand_weight": 1.0,
      "specialty_share": null
    }
  ],
  "rules": {
    "bag_kg": 60,
    "round_capacity": "each round a team may have at most ceil(factor × its total bags ÷ rounds) bags on offer",
    "round_capacity_factor": 1.5,
    "salvage_ratio": 0.8,
    "true_value_formula": "TV = P0 × G(d) × M(m) × S(s)",
    "p0": 120.0,
    "defect_steps": [
      {
        "max_defect_pct": 1.0,
        "step": 1.08,
        "grade": "PREMIUM"
      },
      {
        "max_defect_pct": 3.0,
        "step": 1.04,
        "grade": "GRADE_1"
      },
      {
        "max_defect_pct": 5.0,
        "step": 1.0,
        "grade": "GRADE_2"
      },
      {
        "max_defect_pct": 7.0,
        "step": 0.96,
        "grade": "GRADE_3"
      },
      {
        "max_defect_pct": 15.0,
        "step": 0.85,
        "grade": "BELOW_STANDARD"
      },
   

ฟังก์ชันคิด TV เขียนไว้ให้แล้ว ใช้ตัวเลขจาก `rules` ทั้งหมด ไม่พิมพ์ตัวเลขเอง (ถ้าตลาดเปลี่ยนค่า โค้ดยังถูก)

**TV = P0 × G × M × S** ลองเทียบกับ Excel ของทีม ต้องได้เท่ากัน

In [54]:
rules = cal["rules"]

def grade(d):
    """เกรดและตัวคูณขั้นเกรด จากข้อบกพร่อง d %"""
    for step in rules["defect_steps"]:          # เรียงจากดีไปแย่ เจอขั้นแรกที่ผ่านก็หยุด
        if d <= step["max_defect_pct"]:
            return step["grade"], step["step"]

def moisture_factor(m):
    """ตัวคูณความชื้น M จากความชื้น m %"""
    mo = rules["moisture"]
    if m <= mo["ref"]:                          # แห้งพอ ไม่หัก
        return 1.0
    if m <= mo["hard_limit"]:                   # ชื้นเกิน: หักน้ำหนักน้ำ และหักเพิ่มต่อ % ที่เกิน
        return (1 - m / 100) / (1 - mo["ref"] / 100) - mo["penalty_per_pp"] * (m - mo["ref"])
    return mo["hard_factor"]                    # ชื้นเกินเพดาน ได้ตัวคูณต่ำสุด

def screen_factor(s):
    """ตัวคูณขนาดเมล็ด S · ขนาดนอกตารางใช้ค่าขอบที่ใกล้ที่สุด"""
    table = {int(k): v for k, v in rules["screen_factors"].items()}   # key ใน JSON เป็นข้อความ แปลงเป็นตัวเลข
    return table[min(max(s, min(table)), max(table))]

def true_value(sample):
    """TV บาท/กก. จากผลวัดตัวอย่าง"""
    d, m, s = sample["defect_pct"], sample["moisture_pct"], sample["screen"]
    g = (1 - d / 100) * grade(d)[1]             # G = ส่วนที่ไม่บกพร่อง × ตัวคูณขั้นเกรด
    return rules["p0"] * g * moisture_factor(m) * screen_factor(s)

tv = {}                                         # TV ของแต่ละล็อต ใช้ต่อในข้อ 5, 8, 10
for lot in state["lots"]:
    tv[lot["lot_id"]] = round(true_value(lot["sample"]), 2)
    print(lot["lot_id"], grade(lot["sample"]["defect_pct"])[0], tv[lot["lot_id"]], "บาท/กก.")

LOT-T01-1 GRADE_1 123.43 บาท/กก.
LOT-T01-2 GRADE_1 128.58 บาท/กก.
LOT-T01-3 GRADE_2 114.23 บาท/กก.
LOT-T01-4 GRADE_3 108.96 บาท/กก.
LOT-T01-5 BELOW_STANDARD 80.61 บาท/กก.


## 5. ส่งผลประเมินมูลค่า

ส่งได้ **ครั้งเดียว** และเฉพาะตอน `phase` เป็น `APPRAISAL` · `items` เตรียมไว้ให้แล้ว เติมลงใน body ซึ่งมีรูปแบบ `{"items": [...]}`

สำเร็จได้ `201` และตลาดตอบกลับว่าบันทึกอะไรไว้ · ถ้าส่งซ้ำได้ error `APPRAISAL_ALREADY_SUBMITTED` (ส่งในหน้าเว็บไปแล้วก็ได้ error นี้ ข้ามได้)

In [55]:
where()
# หนึ่งรายการต่อล็อต: lot_id และมูลค่าที่ประเมิน บาท/กก. (grade ใส่หรือไม่ใส่ก็ได้)
items = [{"lot_id": lot_id, "est_value_per_kg": value} for lot_id, value in tv.items()]
print(items)

r = api.post("/v1/appraisals", json={"items":items})   # body เป็น dict ที่มี items อยู่ข้างใน ไม่ใช่ list เปล่า
print(r.status_code)
show(r.json())
# submitted_at  เวลาที่ตลาดรับ (เวลาของเซิร์ฟเวอร์)
# items         สิ่งที่บันทึกไว้ของแต่ละล็อต: grade (null ถ้าไม่ได้ส่ง) และ est_value_per_kg

[T01 · เตรียมตัว (SETUP) · เหลือ 5 วินาที]
[{'lot_id': 'LOT-T01-1', 'est_value_per_kg': 123.43}, {'lot_id': 'LOT-T01-2', 'est_value_per_kg': 128.58}, {'lot_id': 'LOT-T01-3', 'est_value_per_kg': 114.23}, {'lot_id': 'LOT-T01-4', 'est_value_per_kg': 108.96}, {'lot_id': 'LOT-T01-5', 'est_value_per_kg': 80.61}]
409
{
  "error": {
    "code": "WRONG_PHASE",
    "message": "Appraisals are only accepted during APPRAISAL (now: SETUP).",
    "field": null,
    "hint": "Submit once, after the market starts and before round 1.",
    "allowed": {
      "phase": "SETUP",
      "phase_ends_at": 1794975473.085202
    }
  }
}


## 6. เมื่อมีอะไรผิด ตลาดบอกอะไรกลับมา

ช่องนี้ตั้งใจส่งข้อเสนอผิด: 100 กก. ไม่ใช่ทวีคูณของกระสอบ (60 กก.) อ่านอย่างเดียว รันได้เลย

- ตอนนี้รอบยังไม่เปิด จะได้ `409` และ `WRONG_PHASE` (ตลาดตรวจช่วงเวลาก่อน)
- ถ้ารันตอนรอบเปิด จะได้ `400` และ `NOT_MULTIPLE_OF_BAG`

ทุก error มีรูปแบบเดียวกัน อยู่ใต้ `error`:

| ช่อง | ความหมาย |
|---|---|
| `code` | ชื่อ error ให้โปรแกรมอ่าน เช่น `WRONG_PHASE` |
| `message` | ผิดอะไร (ภาษาอังกฤษ) |
| `field` | ช่องไหนผิด เช่น `kg` · `null` ถ้าไม่ใช่ช่องใดช่องหนึ่ง |
| `hint` | ควรทำอย่างไร |
| `allowed` | ค่าที่ใช้ได้ เช่น `nearest: [60, 120]` หรือช่วงตอนนี้และเวลาที่จะจบ |

In [56]:
where()
lot_id = state["lots"][0]["lot_id"]
r = api.post("/v1/offers", json={"rfq_id": "R1-01", "lot_id": lot_id, "kg": 100, "price_per_kg": 120})
print(r.status_code)              # 4xx = คำขอผิด
show(r.json())
print("code:", r.json()["error"]["code"])   # โปรแกรมเช็กช่องนี้เพื่อตัดสินใจว่าจะแก้อย่างไร

[T01 · เตรียมตัว (SETUP) · เหลือ 4 วินาที]
409
{
  "error": {
    "code": "WRONG_PHASE",
    "message": "Offers are only accepted while a round is open (now: SETUP).",
    "field": null,
    "hint": "Poll GET /v1/state until phase is ROUND_OPEN.",
    "allowed": {
      "phase": "SETUP",
      "phase_ends_at": 1794975473.085202
    }
  }
}
code: WRONG_PHASE


## 7. ดูคำขอซื้อรอบนี้ แล้วหาว่าล็อตไหนผ่านสเปก

รันตอน `phase` เป็น `ROUND_OPEN` (ถ้ายังไม่ถึง รอแล้วรันใหม่) · ขอสถานะใหม่ เพราะ RFQ เปลี่ยนทุกรอบ

ช่องของคำขอซื้อ (RFQ) หนึ่งรายการ:

| ช่อง | ความหมาย |
|---|---|
| `rfq_id` | รหัสคำขอซื้อ ใช้ตอนยื่นซอง ใช้ได้เฉพาะรอบนี้ |
| `buyer`, `buyer_id` | ชื่อผู้ซื้อ และรหัส |
| `buyer_type` | ประเภทผู้ซื้อ วันนี้มี `COMMERCIAL` และ `INSTANT` |
| `kg`, `bags` | ต้องการกี่ กก. / กี่กระสอบ |
| `spec` | สเปกขั้นต่ำ: ข้อบกพร่องไม่เกิน `max_defect_pct` · ความชื้นไม่เกิน `max_moisture_pct` · เมล็ดไม่เล็กกว่า `min_screen` (`null` = ไม่กำหนด) |

In [57]:
state = where()                   # where() คืนสถานะทั้งหมดด้วย (เหมือน api.get("/v1/state").json())
show(state["round"])              # รอบที่เท่าไร จากทั้งหมดกี่รอบ
if state["rfqs"]:
    show(state["rfqs"][0])        # คำขอซื้อรายการแรกแบบเต็ม

def passes(sample, spec):
    """ผลวัดตัวอย่างผ่านสเปกของผู้ซื้อทุกข้อไหม"""
    return (sample["defect_pct"] <= spec["max_defect_pct"]
            and sample["moisture_pct"] <= spec["max_moisture_pct"]
            and (spec["min_screen"] is None or sample["screen"] >= spec["min_screen"]))

def overview(state):
    """สรุปสั้นๆ ที่ต้องใช้ตอนยื่นซอง: ล็อตของเรา (เหลือกี่กระสอบ · TV) และคำขอซื้อรอบนี้ (ล็อตไหนผ่านสเปก)"""
    print("ล็อตของเรา:")
    for lot in state["lots"]:
        print(" ", lot["lot_id"], "เหลือยื่นได้", lot["kg_available"] // 60, "กระสอบ · TV", tv[lot["lot_id"]], "บาท/กก.")
    print("คำขอซื้อรอบนี้:")
    for q in state["rfqs"]:
        ok = [lot["lot_id"] for lot in state["lots"] if lot["kg_available"] >= 60 and passes(lot["sample"], q["spec"])]
        print(" ", q["rfq_id"], q["buyer"], q["buyer_type"], q["bags"], "กระสอบ", "· ล็อตที่ผ่าน:", ok)
    if not state["rfqs"]:
        print("  (ยังไม่มี รอ phase เป็น ROUND_OPEN แล้วรันใหม่)")

overview(state)   # ข้อ 8 และ 10 เรียกซ้ำ จะได้ไม่ต้องเลื่อนขึ้นมาดู

[T01 · เตรียมตัว (SETUP) · เหลือ 4 วินาที]
null
ล็อตของเรา:
  LOT-T01-1 เหลือยื่นได้ 4 กระสอบ · TV 123.43 บาท/กก.
  LOT-T01-2 เหลือยื่นได้ 2 กระสอบ · TV 128.58 บาท/กก.
  LOT-T01-3 เหลือยื่นได้ 5 กระสอบ · TV 114.23 บาท/กก.
  LOT-T01-4 เหลือยื่นได้ 8 กระสอบ · TV 108.96 บาท/กก.
  LOT-T01-5 เหลือยื่นได้ 2 กระสอบ · TV 80.61 บาท/กก.
คำขอซื้อรอบนี้:
  (ยังไม่มี รอ phase เป็น ROUND_OPEN แล้วรันใหม่)


## 8. ยื่นซองเสนอขาย 1 รายการ

ตั้ง `RFQ_ID`, `LOT_ID`, `BAGS`, `MARKUP` จากผลข้อ 7 · ราคา = MARKUP × TV · **อย่าต่ำกว่า 0.80** เพราะขายไม่หมดระบบยังให้ 0.80 × มูลค่าจริง

เติม body ให้ครบ 4 ช่อง (`kg` ต้องเป็น กก. ไม่ใช่กระสอบ) แล้วรันเพื่อยื่นซอง · สำเร็จได้ `201` และ `status` เป็น `OPEN`

In [58]:
state = where()
overview(state)                         # rfq_id และ lot_id ที่ใช้ได้ตอนนี้
RFQ_ID = state["rfqs"][0]["rfq_id"]     # หรือพิมพ์เอง เช่น "R1-03"
LOT_ID = state["lots"][0]["lot_id"]
BAGS = 2
MARKUP = 1.00

offer = {
    "rfq_id": "R3-01",                            # ขายให้คำขอซื้อไหน
    "lot_id": "LOT-T01-2",                            # ขายล็อตไหน
    "kg": 2,                           # กี่ กก. ต้องเต็มกระสอบ
    "price_per_kg": round(MARKUP * tv[LOT_ID], 1),   # ราคา บาท/กก.
}
r = api.post("/v1/offers", json=offer)
print(r.status_code)
show(r.json())
# offer_id     รหัสซอง ใช้แก้หรือยกเลิกซองนี้
# round_id     ซองนี้ของรอบไหน (หมดอายุเมื่อรอบปิด)
# status       OPEN = รอจับคู่ · หลังรอบปิดเปลี่ยนเป็นผล (ข้อ 9)
# filled_kg    ขายได้แล้วกี่ กก. (ตอนนี้ 0)

[T01 · เตรียมตัว (SETUP) · เหลือ 4 วินาที]
ล็อตของเรา:
  LOT-T01-1 เหลือยื่นได้ 4 กระสอบ · TV 123.43 บาท/กก.
  LOT-T01-2 เหลือยื่นได้ 2 กระสอบ · TV 128.58 บาท/กก.
  LOT-T01-3 เหลือยื่นได้ 5 กระสอบ · TV 114.23 บาท/กก.
  LOT-T01-4 เหลือยื่นได้ 8 กระสอบ · TV 108.96 บาท/กก.
  LOT-T01-5 เหลือยื่นได้ 2 กระสอบ · TV 80.61 บาท/กก.
คำขอซื้อรอบนี้:
  (ยังไม่มี รอ phase เป็น ROUND_OPEN แล้วรันใหม่)


IndexError: list index out of range

ข้อเสนอที่ส่งด้วย Python ขึ้นในหน้าเว็บด้วย เพราะเป็นตลาดเดียวกัน · ช่องนี้ดู **ข้อเสนอที่เปิดอยู่** คำตอบมี `round_id` และรายการ `offers` แต่ละรายการหน้าตาเหมือนคำตอบตอนยื่น

In [ ]:
where()
res = api.get("/v1/offers").json()
print("รอบ", res["round_id"])
for o in res["offers"]:
    print(o["offer_id"], o["rfq_id"], o["lot_id"], o["kg"] // 60, "กระสอบ", o["price_per_kg"], o["status"])

## 9. ดูผลรอบล่าสุด

รันหลังรอบปิด · คำตอบมีสองส่วน:

**`my_offers`** ซองของเราพร้อมผล: `ACCEPTED` ขายได้ทั้งหมด · `PARTIAL` บางส่วน · `ABOVE_LIMIT` แพงเกินที่ผู้ซื้อรับได้ · `OUTBID` แพ้ข้อเสนอที่คุ้มกว่า · `SPEC_MISMATCH` คุณภาพจริงไม่ผ่านสเปก · `filled_kg` ขายได้กี่ กก.

**`market_summary`** ภาพรวมทั้งตลาดของแต่ละคำขอซื้อ (ไม่บอกว่าใครขาย):

| ช่อง | ความหมาย |
|---|---|
| `wanted_kg`, `bought_kg` | ผู้ซื้ออยากได้กี่ กก. ซื้อจริงกี่ กก. |
| `offers` | มีกี่ซองยื่นให้คำขอนี้ (ทุกทีมรวมบอต) |
| `price_min`, `price_avg`, `price_max` | ราคาของซองที่ขายได้: ต่ำสุด · เฉลี่ยถ่วงตาม กก. · สูงสุด (`null` ถ้าไม่มีใครขายได้) |

ใช้ `market_summary` ดูว่าผู้ซื้อแต่ละประเภทยอมจ่ายราวไหน แล้วปรับ MARKUP ในข้อ 10

In [ ]:
where()

def last_results():
    """ผลรอบล่าสุด: ซองของเรา และภาพรวมทั้งตลาด (เรียก GET /v1/results)"""
    res = api.get("/v1/results").json()
    print("รอบ", res["round_id"], "·", res["kind"])          # kind: real = นับผล
    for o in res["my_offers"]:
        print(o["rfq_id"], o["lot_id"], o["kg"] // 60, "กระสอบ", o["price_per_kg"], "→", o["status"], "ขายได้", o["filled_kg"] // 60)
    print()
    print("ทั้งตลาด:")
    for row in res["market_summary"]:
        print(row["rfq_id"], row["buyer_type"], "ขอ", row["wanted_kg"] // 60, "ซื้อ", row["bought_kg"] // 60, "กระสอบ",
              "·", row["offers"], "ซอง · ราคาที่ขายได้", row["price_min"], "–", row["price_max"], "เฉลี่ย", row["price_avg"])
    return res

res = last_results()   # ข้อ 10 เรียกซ้ำก่อนตั้ง MARKUP

[T01 · รอบเปิดรับข้อเสนอ (ROUND_OPEN) · รอบ 4/8 · เหลือ 79 วินาที]
รอบ R3 · real

ทั้งตลาด:
R3-01 COMMERCIAL ขอ 14 ซื้อ 12 กระสอบ · 7 ซอง · ราคาที่ขายได้ 117.42 – 126.49 เฉลี่ย 121.85
R3-02 SPECIALTY ขอ 3 ซื้อ 3 กระสอบ · 4 ซอง · ราคาที่ขายได้ 126.49 – 137.79 เฉลี่ย 130.26
R3-03 INSTANT ขอ 6 ซื้อ 6 กระสอบ · 14 ซอง · ราคาที่ขายได้ 82.59 – 100.3 เฉลี่ย 92.05
R3-04 SPECIALTY ขอ 2 ซื้อ 0 กระสอบ · 0 ซอง · ราคาที่ขายได้ None – None เฉลี่ย None
R3-05 SPECIALTY ขอ 3 ซื้อ 3 กระสอบ · 2 ซอง · ราคาที่ขายได้ 137.09 – 137.09 เฉลี่ย 137.09
R3-06 SPECIALTY ขอ 4 ซื้อ 0 กระสอบ · 1 ซอง · ราคาที่ขายได้ None – None เฉลี่ย None
R3-07 COMMERCIAL ขอ 16 ซื้อ 16 กระสอบ · 10 ซอง · ราคาที่ขายได้ 114.61 – 126.49 เฉลี่ย 121.05
R3-08 INSTANT ขอ 26 ซื้อ 26 กระสอบ · 37 ซอง · ราคาที่ขายได้ 80.78 – 112.57 เฉลี่ย 99.7
R3-09 COMMERCIAL ขอ 6 ซื้อ 0 กระสอบ · 0 ซอง · ราคาที่ขายได้ None – None เฉลี่ย None


## 10. รอบถัดไป: ยื่นทุก RFQ ในครั้งเดียว

รอบต่อๆ ไปทำซ้ำข้อ 7 → 8 → 9 ได้ แต่จะช้า ช่องนี้วนยื่นให้ทุก RFQ ที่มีล็อตผ่านสเปก
ทุกครั้งที่รอบเปิด: รันช่องแรกดูข้อมูลล่าสุด (ไม่ต้องเลื่อนขึ้นไปดูข้อ 7 และ 9) → ตั้ง MARKUP แยกตามประเภทผู้ซื้อในช่องที่สอง แล้วเติม body ของซอง → รันช่องที่สอง

In [ ]:
state = where()
overview(state)        # ล็อตที่เหลือ TV และ rfq_id ของรอบนี้ (ข้อ 7)
print()
res = last_results()   # รอบที่แล้วแต่ละประเภทขายได้ราคาเท่าไร (ข้อ 9) · ราคาเฉลี่ย ÷ TV ของล็อต ≈ MARKUP ที่ขายได้

[T01 · รอบเปิดรับข้อเสนอ (ROUND_OPEN) · รอบ 5/8 · เหลือ 90 วินาที]
ล็อตของเรา:
  LOT-T01-1 เหลือยื่นได้ 6 กระสอบ · TV 94.76 บาท/กก.
  LOT-T01-2 เหลือยื่นได้ 5 กระสอบ · TV 126.49 บาท/กก.
  LOT-T01-3 เหลือยื่นได้ 6 กระสอบ · TV 90.76 บาท/กก.
  LOT-T01-4 เหลือยื่นได้ 5 กระสอบ · TV 121.06 บาท/กก.
  LOT-T01-5 เหลือยื่นได้ 4 กระสอบ · TV 110.22 บาท/กก.
  LOT-T01-6 เหลือยื่นได้ 0 กระสอบ · TV 121.89 บาท/กก.
คำขอซื้อรอบนี้:
  R5-01 2 บางกอกบลูม SPECIALTY 2 กระสอบ · ล็อตที่ผ่าน: ['LOT-T01-2']
  R5-02 5 อินสแตนท์ไทย INSTANT 23 กระสอบ · ล็อตที่ผ่าน: ['LOT-T01-1', 'LOT-T01-2', 'LOT-T01-3', 'LOT-T01-4', 'LOT-T01-5']
  R5-03 6 ทรีอินวันฟู้ดส์ INSTANT 9 กระสอบ · ล็อตที่ผ่าน: ['LOT-T01-1', 'LOT-T01-2', 'LOT-T01-3', 'LOT-T01-4', 'LOT-T01-5']
  R5-04 2 บางกอกบลูม SPECIALTY 5 กระสอบ · ล็อตที่ผ่าน: ['LOT-T01-2']
  R5-05 4 ชุมพรคั่วบด COMMERCIAL 20 กระสอบ · ล็อตที่ผ่าน: ['LOT-T01-2', 'LOT-T01-4']
  R5-06 3 สยามคอมเมิร์ซ COMMERCIAL 16 กระสอบ · ล็อตที่ผ่าน: ['LOT-T01-2', 'LOT-T01-4']
  R5-07 1 ดอยช้างโรสเตอร์ S

In [ ]:
# ตัวคูณ TV ของผู้ซื้อแต่ละประเภท: ราคา = MARKUP × TV เช่น 1.05 = แพงกว่า TV 5% · ไม่ต่ำกว่า 0.80
MARKUP_BY_TYPE = {"COMMERCIAL": ___, "INSTANT": ___}

state = where()
for q in state["rfqs"]:
    ok = [lot for lot in state["lots"] if lot["kg_available"] >= 60 and passes(lot["sample"], q["spec"])]
    if not ok:
        continue                                          # ไม่มีล็อตผ่านสเปก ข้ามคำขอนี้
    lot = ok[0]
    bags = min(q["bags"], lot["kg_available"] // 60)      # ไม่เกินที่ผู้ซื้อขอ และไม่เกินที่ล็อตเหลือ
    price = round(MARKUP_BY_TYPE[q["buyer_type"]] * tv[lot["lot_id"]], 1)
    offer = {                                             # 4 ช่องเหมือนข้อ 8
        "rfq_id": ,
        "lot_id": ___,
        "kg": ___,
        "price_per_kg": ___,
    }
    r = api.post("/v1/offers", json=offer)
    print(q["rfq_id"], lot["lot_id"], bags, "กระสอบ", price, "→", r.status_code, r.json().get("status") or r.json()["error"]["code"])
    lot["kg_available"] -= bags * 60      # กันยื่นกระสอบเดียวกันซ้ำในรอบเดียว

## 11. จบ 6 รอบ: ดูมูลค่าจริง

รันเมื่อ `phase` เป็น `CLOSED` · เหมือนปุ่ม **"ดูมูลค่าจริง"** ในหน้าเว็บ (ก่อนปิดตลาดได้ error `NOT_AVAILABLE`)

คำตอบมีสองส่วน: `lots` เฉลยของแต่ละล็อต และ `score` คะแนนของเกมนี้

| ช่อง | ความหมาย |
|---|---|
| `sample` / `true` | ผลวัดตัวอย่างที่เห็น / คุณภาพจริงทั้งล็อต เทียบกันดูว่าตัวอย่างคลาดไปเท่าไร |
| `true_grade`, `true_value_per_kg` | เกรดจริง และมูลค่าจริง บาท/กก. |
| `your_estimate` | ผลประเมินที่ทีมส่ง (`null` ถ้าไม่ได้ส่ง) |
| `score` | `pr` อัตราราคาที่ขายได้เทียบมูลค่าจริง · `mape` ประเมินคลาดเฉลี่ย · `pr_score`, `appraisal_score`, `total` คะแนน |

In [ ]:
where()
rv = api.get("/v1/sandbox/reveal").json()
show(rv["lots"][0])               # เฉลยล็อตแรกแบบเต็ม
for lot in rv["lots"]:
    est = (lot["your_estimate"] or {}).get("est_value_per_kg")
    print(lot["lot_id"], "TV จริง", lot["true_value_per_kg"], "· เราประเมิน", est)

sc = rv["score"]
show(sc)
print("PR", sc["pr"], "· คะแนนประเมิน", round(sc["appraisal_score"], 1), "· คะแนนรวม", round(sc["total"], 1))

## ก้าวต่อไป (พรุ่งนี้): ให้โปรแกรมทำเองทุกรอบ

ทุกอย่างข้างบนคือสิ่งที่ agent ทำ ต่างกันแค่ agent **วนซ้ำเอง** ทุกรอบ:

```
ทุก 1–2 วินาที:
    ดูสถานะ (GET /v1/state)
    ถ้าถึงช่วงประเมินและยังไม่ส่ง → คำนวณ TV แล้วส่ง
    ถ้ารอบใหม่เปิด → ตัดสินใจว่าจะเสนอใคร เท่าไร ราคาเท่าไร → ส่ง
    ถ้ารอบปิด → อ่านผล จำไว้ใช้รอบหน้า
```

ส่วน "ตัดสินใจ" คือที่ใช้ทั้งกฎ (Python) และ LLM เรียนกันวันที่ 18 · คู่มือ API ครบทุกคำขอและทุก error อยู่ที่ `คู่มือ-API.md` ใน repo starter
**ห้ามเรียก API ถี่เกิน 5 ครั้งต่อวินาที** (จะได้ error `RATE_LIMITED`)